In [1]:
import os

# Safely list files under ./input (single quick pass)
print('\n'.join([
    os.path.join(root, f)
    for root, _, files in os.walk('./input')
    for f in files
]))

import numpy as np
import pandas as pd

./input/titanic.zip
./input/gender_submission.csv
./input/test.csv
./input/train.csv
./input/titanic.zip:Zone.Identifier


In [2]:
train =pd.read_csv('./input/train.csv')
test = pd.read_csv('./input/test.csv')
gender_submission = pd.read_csv('./input/gender_submission.csv')

In [3]:
data = pd.concat([train,test],sort = False)
data['Sex']=data['Sex'].replace(['male','female'],[0,1])
#male,femaleを０１に変換

/tmp/ipykernel_12240/3092714075.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Sex']=data['Sex'].replace(['male','female'],[0,1])


In [4]:
data['Fare']=data['Fare'].fillna(np.mean(data['Fare']))
data['Embarked']= data['Embarked'].fillna('S').map({'S':0,'C':1,'Q':2}).astype(int)
age_ave = data['Age'].mean()
age_std = data['Age'].std()
data['Age'] = data['Age'].fillna(np.random.randint(age_ave - age_std,age_ave+ age_std))
#np.ranom.randint(a,b)はa以上b未満の整数をランダム生成
#data[].fillna(implace=True)みたいにやるのは古いのでdata[] = data[].fillnaにする

In [5]:
delete_columns = ['Name', 'PassengerId', 'Parch', 'Ticket', 'Cabin']
data = data.drop(delete_columns, axis=1)
#いらない特徴量を消す

In [6]:
train = data[:len(train)]
test = data[len(train):]
#行数で分離

In [7]:
y_train = train['Survived']
X_train = train.drop('Survived', axis=1)
X_test = test.drop('Survived', axis=1)
#最終結果だけ分離

In [8]:
categorical_features = ['Embarked', 'Pclass', 'Sex']
#カテゴリ変数だと宣言しないと連続数とみなされる。

In [12]:
import lightgbm as lgb
from sklearn.model_selection import KFold


y_preds = []
models = []
oof_train = np.zeros((len(X_train),))
cv = KFold(n_splits=5, shuffle=True, random_state=0)

categorical_features = ['Embarked', 'Pclass', 'Sex']

params = {
    'objective': 'binary',
    'max_bin': 300,
    'learning_rate': 0.05,
    'num_leaves': 40
}

for fold_id, (train_index, valid_index) in enumerate(cv.split(X_train)):
    X_tr = X_train.loc[train_index, :]
    X_val = X_train.loc[valid_index, :]
    y_tr = y_train[train_index]
    y_val = y_train[valid_index]

    lgb_train = lgb.Dataset(X_tr, y_tr,
                                             categorical_feature=categorical_features)
    lgb_eval = lgb.Dataset(X_val, y_val, reference=lgb_train,
                                            categorical_feature=categorical_features)

    model = lgb.train(params, lgb_train,
                                   valid_sets=[lgb_train, lgb_eval],
                                   callbacks=[lgb.log_evaluation(10), lgb.early_stopping(10)],
                                   num_boost_round=1000)


    oof_train[valid_index] = model.predict(X_val, num_iteration=model.best_iteration)
    y_pred = model.predict(X_test, num_iteration=model.best_iteration)

    y_preds.append(y_pred)
    models.append(model)

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003036 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 195
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 10 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

In [13]:
pd.DataFrame(oof_train).to_csv('oof_train_kfold.csv', index=False)

scores = [
    m.best_score['valid_1']['binary_logloss'] for m in models
]
score = sum(scores) / len(scores)
print('===CV scores===')
print(scores)
print(score)

===CV scores===
[np.float64(0.36572168267259164), np.float64(0.42797933275778055), np.float64(0.36951704083446457), np.float64(0.4325961445633855), np.float64(0.4464503618032553)]
0.40845291252629556


In [14]:
from sklearn.metrics import accuracy_score


y_pred_oof = (oof_train > 0.5).astype(int)
accuracy_score(y_train, y_pred_oof)

0.8372615039281706

In [ ]:
y_pred = (y_pred > 0.5).astype(int)
y_pred[:10]

In [15]:
len(y_preds)

5

In [16]:
y_preds[0][:10]

array([0.11027976, 0.33129929, 0.03924897, 0.32695304, 0.36974635,
       0.6965518 , 0.68522627, 0.11771262, 0.86292527, 0.03126738])

In [17]:
y_sub = sum(y_preds) / len(y_preds)
y_sub = (y_sub > 0.5).astype(int)
y_sub[:10]

array([0, 0, 0, 0, 0, 0, 1, 0, 1, 0])

In [ ]:
y_sub['Survived'] = y_sub
y_sub.to_csv('submission_lightgbm_kfold.csv', index=False)

y_sub.head()

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [ ]:
lgb.plot_importance(model, importance_type='gain')